In [14]:
import os
import time
import gc
import json
import joblib
import psutil
import numpy as np
import pandas as pd
import torch
import sklearn
from transformers import AutoTokenizer, AutoModelForCausalLM
print("Resource benchmarking started.")

Resource benchmarking started.


In [15]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
process = psutil.Process(os.getpid())

def get_ram_mb():
    """
    Return current process RAM usage in MB.
    """
    return process.memory_info().rss / (1024 ** 2)


def benchmark_time(function, *args, **kwargs):
    """
    Measure execution time of a function.
    """
    start = time.perf_counter()

    result = function(*args, **kwargs)

    elapsed = time.perf_counter() - start

    return result, elapsed


print("Benchmark helpers ready.")

Benchmark helpers ready.


In [3]:
customer_df = pd.read_csv(
    "processed/telco_eda_processed.csv"
)

print("Customer dataset:", customer_df.shape)

Customer dataset: (7043, 26)


In [4]:
start_ram = get_ram_mb()
start_time = time.perf_counter()

preprocessor = joblib.load(
    "../models/preprocessor.pkl"
)

churn_model = joblib.load(
    "../models/improved_logistic_regression.pkl"
)

model_loading_time = time.perf_counter() - start_time
end_ram = get_ram_mb()

model_ram_increase = end_ram - start_ram

print("Model loading time:",
      round(model_loading_time, 4),
      "seconds")

print("Approximate RAM increase:",
      round(model_ram_increase, 2),
      "MB")

Model loading time: 0.0948 seconds
Approximate RAM increase: 3.79 MB


In [5]:
def engineer_customer_features(customer_df):

    customer_df = customer_df.copy()

    customer_df["TotalCharges"] = pd.to_numeric(
        customer_df["TotalCharges"],
        errors="coerce"
    )

    customer_df["TotalCharges"] = customer_df["TotalCharges"].fillna(
        customer_df["TotalCharges"].median()
    )

    customer_df["NewCustomer"] = (
        customer_df["tenure"] <= 6
    ).astype(int)

    customer_df["LongTermCustomer"] = (
        customer_df["tenure"] >= 48
    ).astype(int)

    customer_df["ChargeToTenure"] = (
        customer_df["TotalCharges"] /
        (customer_df["tenure"] + 1)
    )

    monthly_threshold = customer_df["MonthlyCharges"].quantile(0.75)

    customer_df["HighMonthlyCharge"] = (
        customer_df["MonthlyCharges"] >= monthly_threshold
    ).astype(int)

    customer_df["HighValueHighRisk"] = (
        (customer_df["MonthlyCharges"] >= monthly_threshold)
        &
        (customer_df["tenure"] <= 6)
    ).astype(int)

    return customer_df

In [6]:
customer_df = pd.read_csv(
    "processed/telco_eda_processed.csv"
)

customer_df = engineer_customer_features(customer_df)

In [7]:
sample_customer = customer_df.iloc[[0]].copy()

sample_customer_id = sample_customer["customerID"].iloc[0]

X_sample = sample_customer.drop(
    columns=["customerID", "Churn"],
    errors="ignore"
)

X_sample_processed = preprocessor.transform(
    X_sample
)

print("Sample customer:", sample_customer_id)
print("Processed feature shape:",
      X_sample_processed.shape)

Sample customer: 7590-VHVEG
Processed feature shape: (1, 65)


In [8]:
start = time.perf_counter()

ml_probability = churn_model.predict_proba(
    X_sample_processed
)[0][1]

ml_inference_time = time.perf_counter() - start

print(
    "ML churn probability:",
    round(float(ml_probability), 4)
)

print(
    "ML inference time:",
    round(ml_inference_time, 6),
    "seconds"
)

ML churn probability: 0.7568
ML inference time: 0.002604 seconds


/home/aximsoft/nltk-env/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [9]:
from sentence_transformers import SentenceTransformer

embedding_model_name = "all-MiniLM-L6-v2"

start_ram = get_ram_mb()
start_time = time.perf_counter()

embedding_model = SentenceTransformer(
    embedding_model_name
)

embedding_loading_time = time.perf_counter() - start_time
embedding_ram_increase = get_ram_mb() - start_ram

print(
    "Embedding model loading time:",
    round(embedding_loading_time, 4),
    "seconds"
)

print(
    "Approximate RAM increase:",
    round(embedding_ram_increase, 2),
    "MB"
)

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 8629.39it/s]


Embedding model loading time: 7.5851 seconds
Approximate RAM increase: 66.46 MB


In [10]:
sample_text = """
Customer is using a month-to-month contract
and has a relatively high monthly charge.
"""

start = time.perf_counter()

sample_embedding = embedding_model.encode(
    [sample_text],
    normalize_embeddings=True
)

embedding_inference_time = time.perf_counter() - start

print(
    "Embedding shape:",
    sample_embedding.shape
)

print(
    "Embedding inference time:",
    round(embedding_inference_time, 6),
    "seconds"
)

Embedding shape: (1, 384)
Embedding inference time: 0.029989 seconds


In [11]:
import faiss

faiss_index_path = "../vector_store/knowledge_faiss.index"

if os.path.exists(faiss_index_path):

    knowledge_index = faiss.read_index(
        faiss_index_path
    )

    print(
        "FAISS index loaded."
    )

    print(
        "Number of vectors:",
        knowledge_index.ntotal
    )

else:

    knowledge_index = None

    print(
        "Knowledge FAISS index not found."
    )

FAISS index loaded.
Number of vectors: 6


In [22]:
if knowledge_index is not None:
 
    query_embedding = np.asarray(
        sample_embedding,
        dtype="float32"
    )
 
    start = time.perf_counter()
 
    distances, indices = knowledge_index.search(
        query_embedding,
        3
    )
 
    faiss_search_time = time.perf_counter() - start
 
    print(
        "FAISS search time:",
        round(faiss_search_time, 6),
        "seconds"
    )
 
    print(
        "Retrieved indices:",
        indices[0]
    )
 
    print(
        "Similarity scores:",
        distances[0]
    )
 
else:
 
    faiss_search_time = np.nan

FAISS search time: 0.011292 seconds
Retrieved indices: [0 2 3]
Similarity scores: [0.5256202  0.45077538 0.35651824]


In [15]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

llm.eval()

print("Local LLM loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:05<00:00, 60.64it/s]


Local LLM loaded.


In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

start_ram = get_ram_mb()
start_time = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

llm_model.eval()

llm_loading_time = time.perf_counter() - start_time
llm_ram_increase = get_ram_mb() - start_ram

print(
    "LLM loading time:",
    round(llm_loading_time, 3),
    "seconds"
)

print(
    "Approximate LLM RAM increase:",
    round(llm_ram_increase, 2),
    "MB"
)

Loading weights: 100%|███████████████████████| 338/338 [00:07<00:00, 45.44it/s]


LLM loading time: 10.698 seconds
Approximate LLM RAM increase: 3385.04 MB


In [17]:
benchmark_prompt = """
You are a customer intelligence assistant.

Give a short explanation of why a customer with
a month-to-month contract and high monthly charges
may require retention attention.

Answer using only the information provided.
"""

inputs = tokenizer(
    benchmark_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

start = time.perf_counter()

with torch.no_grad():

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

llm_inference_time = time.perf_counter() - start

generated_tokens = outputs[
    0
][inputs["input_ids"].shape[1]:]

llm_answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("LLM output:")
print(llm_answer)

print(
    "\nLLM inference time:",
    round(llm_inference_time, 3),
    "seconds"
)

LLM output:
A customer with a month-to-month contract and high monthly charges may require retention attention due to potential financial instability or dissatisfaction. High monthly charges could indicate that the customer is struggling financially, which might lead them to seek alternative services or providers. Additionally, if this customer has recently signed up for a new service or product, their high charges might suggest they are not satisfied with the current offering, prompting retention efforts

LLM inference time: 15.956 seconds


In [18]:
benchmark_customer_id = customer_df[
    "customerID"
].iloc[0]

benchmark_question = (
    "Explain this customer's churn risk "
    "and relevant retention information."
)

print("Customer:", benchmark_customer_id)
print("Question:", benchmark_question)

Customer: 7590-VHVEG
Question: Explain this customer's churn risk and relevant retention information.


In [19]:
start_ram = get_ram_mb()
start_time = time.perf_counter()

try:

    agent_result = simple_customer_agent(
        benchmark_question,
        benchmark_customer_id
    )

    end_to_end_status = "Success"

except Exception as e:

    agent_result = {
        "answer": str(e),
        "selected_tools": []
    }

    end_to_end_status = "Failed"

end_to_end_time = time.perf_counter() - start_time
end_to_end_ram = get_ram_mb() - start_ram

print(
    "End-to-end status:",
    end_to_end_status
)

print(
    "End-to-end response time:",
    round(end_to_end_time, 3),
    "seconds"
)

print(
    "Approximate additional RAM:",
    round(end_to_end_ram, 2),
    "MB"
)

print("\nAgent answer:")
print(agent_result["answer"])

End-to-end status: Failed
End-to-end response time: 0.0 seconds
Approximate additional RAM: 0.0 MB

Agent answer:
name 'simple_customer_agent' is not defined


In [23]:
benchmark_results = [
    {
        "Component": "Churn Model Loading",
        "TimeSeconds": model_loading_time,
        "RAMIncreaseMB": model_ram_increase
    },
    {
        "Component": "ML Inference",
        "TimeSeconds": ml_inference_time,
        "RAMIncreaseMB": np.nan
    },
    {
        "Component": "Embedding Model Loading",
        "TimeSeconds": embedding_loading_time,
        "RAMIncreaseMB": embedding_ram_increase
    },
    {
        "Component": "Embedding Inference",
        "TimeSeconds": embedding_inference_time,
        "RAMIncreaseMB": np.nan
    },
    {
        "Component": "FAISS Search",
        "TimeSeconds": faiss_search_time,
        "RAMIncreaseMB": np.nan
    },
    {
        "Component": "LLM Loading",
        "TimeSeconds": llm_loading_time,
        "RAMIncreaseMB": llm_ram_increase
    },
    {
        "Component": "LLM Inference",
        "TimeSeconds": llm_inference_time,
        "RAMIncreaseMB": np.nan
    },
    {
        "Component": "End-to-End Agent",
        "TimeSeconds": end_to_end_time,
        "RAMIncreaseMB": end_to_end_ram
    }
]

benchmark_df = pd.DataFrame(
    benchmark_results
)

benchmark_df

,Component,TimeSeconds,RAMIncreaseMB
0,Churn Model Loading,0.094845,3.792969
1,ML Inference,0.002604,NaN
2,Embedding Model Loading,7.585053,66.464844
3,Embedding Inference,0.029989,NaN
4,FAISS Search,0.011292,NaN
5,LLM Loading,10.697863,3385.035156
6,LLM Inference,15.955919,NaN
7,End-to-End Agent,0.000254,0.000000


In [24]:
benchmark_df.to_csv(
    "../data/processed/resource_benchmark_results.csv",
    index=False
)

print(
    "Benchmark results saved successfully."
)

Benchmark results saved successfully.


In [25]:
print("=" * 65)
print("RESOURCE BENCHMARKING")
print("=" * 65)

print(
    "\nChurn Model Loading:",
    round(model_loading_time, 4),
    "seconds"
)

print(
    "ML Inference:",
    round(ml_inference_time, 6),
    "seconds"
)

print(
    "Embedding Model Loading:",
    round(embedding_loading_time, 4),
    "seconds"
)

print(
    "Embedding Inference:",
    round(embedding_inference_time, 6),
    "seconds"
)

print(
    "FAISS Search:",
    round(faiss_search_time, 6),
    "seconds"
)

print(
    "LLM Loading:",
    round(llm_loading_time, 3),
    "seconds"
)

print(
    "LLM Inference:",
    round(llm_inference_time, 3),
    "seconds"
)

print(
    "End-to-End Agent:",
    round(end_to_end_time, 3),
    "seconds"
)

print(
    "\nCurrent Process RAM:",
    round(get_ram_mb(), 2),
    "MB"
)

print("=" * 65)
print("Notebook 16 completed.")

RESOURCE BENCHMARKING

Churn Model Loading: 0.0948 seconds
ML Inference: 0.002604 seconds
Embedding Model Loading: 7.5851 seconds
Embedding Inference: 0.029989 seconds
FAISS Search: 0.011292 seconds
LLM Loading: 10.698 seconds
LLM Inference: 15.956 seconds
End-to-End Agent: 0.0 seconds

Current Process RAM: 10148.16 MB
Notebook 16 completed.
